---
## 13. Question F - Original research question

**1. Question.** *How persistent are listings from one day to the next within our daily samples, and are the prices of listings that appear repeatedly stable?*

**2. Why it is relevant.** Persistence shows how fast the visible marketplace turns over and whether the seven daily files can be treated as the same population or as different samples. Price stability tells us whether price changes over a week are a real market feature. Both are directly needed to interpret Question C. They were chosen after seeing the data, because the identifier and the repeated collection are the strongest parts of the dataset (unlike condition for Fashion or location for Furniture).

**3. Variables.** `listing_key` (from the URL), `collection_date`, `category`, `price`.

**4. Method.** Count on how many different days each unique listing appears; compute the overlap between days for Mobile Phones (the only category collected on all seven days); compare first and last prices of repeated listings.

**5. Visualisations.** F1 heatmap of day-to-day overlap (phones) and F2 distribution of the number of days each listing was seen, by category.

In [ ]:
# Q F: How persistent are listings in the daily samples, and are repeated listings' prices stable?
days_seen = df.groupby("listing_key").agg(category=("category", "last"), n_days=("collection_date", "nunique"))
dist = days_seen.groupby(["category", "n_days"]).size().unstack(fill_value=0).reindex(CATS)
dist_pct = dist.div(dist.sum(axis=1), axis=0) * 100
display(dist)
print("Share of each category's unique listings seen on exactly 1 day, %:"); print(dist_pct[1].round(1).to_string())
print("Share seen on 2 or more days, %:"); print((100 - dist_pct[1]).round(1).to_string())

# Overlap between days, Mobile Phones only (the only category collected on all 7 days)
ph_sets = {d: set(g.listing_key) for d, g in df[df.category == "Mobile Phones"].groupby("collection_date")}
dl = sorted(ph_sets)
overlap = pd.DataFrame([[100 * len(ph_sets[a] & ph_sets[b]) / len(ph_sets[a]) for b in dl] for a in dl],
                       index=[d.strftime("%d %b") for d in dl], columns=[d.strftime("%d %b") for d in dl])
display(overlap.round(1))
cons = [100 * len(ph_sets[a] & ph_sets[b]) / len(ph_sets[a]) for a, b in zip(dl[:-1], dl[1:])]
print("Phones: % of one day's listings that are also present the NEXT collected day:", [round(x, 1) for x in cons])

# Price stability of repeated listings
stab = matched.assign(changed=lambda d: d.direction != "No change")
print("\nShare of repeated listings whose price changed, by number of days seen:")
print((stab.groupby("n_days")["changed"].agg(["mean", "size"]).rename(columns={"mean": "share_changed", "size": "listings"}).assign(share_changed=lambda d: (100 * d.share_changed).round(1))).to_string())
print("Share changed by span between first and last observation (days):")
bins = pd.cut(stab["span_days"], [0, 2, 4, 6, 7], labels=["1-2 d", "3-4 d", "5-6 d", "7 d"])
print(stab.groupby(bins)["changed"].agg(["mean", "size"]).assign(mean=lambda d: (100 * d["mean"]).round(1)).to_string())